# Pregunta 2 — Análisis factorial exploratorio

**Métodos Cuantitativos de Investigación en Negocios — Tarea 2 (2026)**

Este notebook documenta cada decisión, fórmula, cálculo, supuesto, resultado e interpretación. Se ejecuta de principio a fin con la base SPSS correspondiente.

In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import pyreadstat
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)
DATA_DIR = Path.cwd()
if not (DATA_DIR / 'Med_Mod.sav').exists():
    DATA_DIR = Path(r'/Users/sbc/projects/PhD-Business-Economics-DEN-UDD/Metodos de Investigacion/Tarea2')
print('Directorio de datos:', DATA_DIR)

Directorio de datos: /Users/sbc/projects/PhD-Business-Economics-DEN-UDD/Metodos de Investigacion/Tarea2


## 1. Objetivo y decisiones analíticas

Se evalúa si los 28 ítems del EECS-R reproducen cuatro dimensiones correlacionadas propuestas por la teoría de Blend: amor por la estadística, entusiasmo por el diseño experimental, amor por la enseñanza y ausencia de habilidades interpersonales.

Se usa **análisis factorial exploratorio de factores comunes**, extracción de máxima verosimilitud y rotación oblicua Oblimin, porque las dimensiones pueden correlacionarse. Los datos perdidos —solo nueve respuestas— se imputan con la mediana del ítem.

In [2]:
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity, calculate_kmo
df,meta=pyreadstat.read_sav(DATA_DIR/'EECS-R.sav',apply_value_formats=False)
items=[f'q{i}' for i in range(1,29)]; X=df[items].copy()
print('Casos:',len(X),'Ítems:',len(items),'Datos perdidos:',int(X.isna().sum().sum()))
X=X.apply(lambda s:s.fillna(s.median()))
for c in items: print(f'{c}: {meta.column_names_to_labels[c]}')

Casos: 239 Ítems: 28 Datos perdidos: 9
q1: Una vez me desperté en un campo de vegetales abrazando un nabo el que erronaemente extraje pensando que era la Mayor Raiz de Roy
q2: Si tuviese una pistola grande le dispararía a todos los estudiantes a los que tengo que enseñarles
q3: Me gusta memorizar valores de la distribucion F
q4: Le tengo un altar a Pearson en mi oficina
q5: Todavía vivo con mi madre y tengo poca preocupacion por mi higiene personal
q6: Enseñarle a otros me hace desear el tragar una botella de cloro porque, comparativamente, el dolor en mi esófago ardiente sería un gran alivio
q7: Ayudarle a otros a entender la suma de cuadrados me provoca gran emocion
q8: Me gustan las condiciones de control
q9: Calculo 3 ANOVAs mentalmente antes de levantarme cada mañana
q10: Podría pasar todo el día explicandole estadísticas a las personas
q11: Me encanta cuando la gente me dice que les ayudé a entender la rotación de factores
q12: La gente se queda dormida apenas abro la boca para h

## 2. Factorabilidad: Bartlett y KMO

Bartlett contrasta que la matriz de correlaciones sea identidad. Se requiere $p<0.05$. KMO cuantifica varianza común; valores superiores a 0.80 son meritorios.

In [3]:
chi,p=calculate_bartlett_sphericity(X); _,kmo=calculate_kmo(X)
print(f'Bartlett: chi-cuadrado={chi:.4f}, gl={28*27//2}, p={p:.4g}')
print(f'KMO global={kmo:.4f}')

Bartlett: chi-cuadrado=3045.3894, gl=378, p=0
KMO global=0.8952


## 3. Número de factores

Se combinan el gráfico de sedimentación, autovalores y análisis paralelo con 1.000 matrices aleatorias. Se retienen los componentes observados cuyo autovalor supera el percentil 95 de los autovalores aleatorios.

In [4]:
R=X.corr().to_numpy(); eig=np.linalg.eigvalsh(R)[::-1]
rng=np.random.default_rng(20260706); re=[]
for _ in range(1000): re.append(np.linalg.eigvalsh(np.corrcoef(rng.normal(size=X.shape),rowvar=False))[::-1])
pa95=np.quantile(re,.95,axis=0); n=int(np.sum(eig>pa95))
print(pd.DataFrame({'Factor':range(1,29),'Autovalor observado':eig,'Percentil 95 aleatorio':pa95}).head(10).round(3).to_string(index=False))
print('\nFactores sugeridos por análisis paralelo:',n)

 Factor  Autovalor observado  Percentil 95 aleatorio
      1                9.020                   1.790
      2                2.743                   1.662
      3                1.707                   1.577
      4                1.505                   1.502
      5                1.158                   1.433
      6                0.976                   1.377
      7                0.916                   1.321
      8                0.829                   1.272
      9                0.791                   1.225
     10                0.737                   1.182

Factores sugeridos por análisis paralelo: 4


## 4. Solución de cuatro factores

Se reporta la matriz patrón. Como reglas orientativas, una carga de $|0.40|$ o más es sustantiva; cargas similares en dos factores indican ítems complejos. La rotación oblicua permite correlación factorial.

In [5]:
fa=FactorAnalyzer(n_factors=4,rotation='oblimin',method='ml'); fa.fit(X)
L=pd.DataFrame(fa.loadings_,index=items,columns=['F1','F2','F3','F4'])
print(L.round(3).to_string())
print('\nVarianza por factor:')
print(pd.DataFrame(np.array(fa.get_factor_variance()).T,columns=['SS cargas','Proporción','Acumulada'],index=['F1','F2','F3','F4']).round(3).to_string())
print('\nCorrelaciones factoriales:')
print(pd.DataFrame(fa.phi_,index=L.columns,columns=L.columns).round(3).to_string())

        F1     F2     F3     F4
q1   0.401 -0.140  0.285  0.340
q2   0.097 -0.329  0.357  0.272
q3   0.116  0.293 -0.048  0.508
q4   0.125  0.165  0.137  0.510
q5   0.056  0.055  0.669 -0.131
q6   0.160 -0.347  0.161  0.264
q7   0.139  0.452 -0.049  0.299
q8   0.575  0.222  0.036  0.242
q9   0.437 -0.114  0.258  0.372
q10  0.417 -0.069  0.178  0.044
q11  0.266  0.457  0.200  0.102
q12 -0.011  0.133  0.269 -0.164
q13  0.560  0.082 -0.033  0.209
q14  0.757  0.028  0.179 -0.337
q15  0.361 -0.017  0.137  0.184
q16  0.741  0.215 -0.024 -0.200
q17  0.788 -0.009 -0.034  0.136
q18  0.232 -0.322  0.404  0.142
q19 -0.055  0.568 -0.177  0.078
q20  0.035  0.450  0.052  0.199
q21  0.471  0.165 -0.040  0.270
q22  0.857 -0.074 -0.021  0.113
q23 -0.094  0.042  0.700  0.066
q24  0.097  0.284  0.127  0.481
q25  0.101  0.533  0.131  0.082
q26  0.278  0.593  0.020 -0.072
q27 -0.020  0.612  0.382  0.068
q28  0.012  0.211  0.533 -0.016

Varianza por factor:
    SS cargas  Proporción  Acumulada
F1      4.322

## 5. Interpretación y confiabilidad

La solución se interpreta considerando conjuntamente contenido y cargas:

- F1: entusiasmo por diseño experimental.
- F2: amor por la enseñanza.
- F3: ausencia de habilidades interpersonales.
- F4: amor por la estadística.

Se calcula alfa con conjuntos conceptualmente coherentes. Algunos ítems presentan cargas cruzadas o débiles, por lo que la teoría se evalúa en grado, no solo contando factores.

In [6]:
def alpha(d):
 k=d.shape[1]; return k/(k-1)*(1-d.var(ddof=1).sum()/d.sum(axis=1).var(ddof=1))
sets={
'Diseño experimental':['q8','q13','q14','q16','q17','q22'],
'Enseñanza':['q7','q10','q11','q19','q20','q25','q26'],
'Habilidades interpersonales deficientes':['q2','q5','q6','q12','q18','q23','q27','q28'],
'Estadística':['q1','q3','q4','q9','q15','q21','q24']}
for nombre,its in sets.items(): print(f'{nombre}: alpha={alpha(X[its]):.4f} ({len(its)} ítems)')

Diseño experimental: alpha=0.8795 (6 ítems)
Enseñanza: alpha=0.7471 (7 ítems)
Habilidades interpersonales deficientes: alpha=0.6908 (8 ítems)
Estadística: alpha=0.8302 (7 ítems)


## 6. Conclusión

Bartlett es significativo ($\chi^2(378)=3045.39$, $p<0.001$) y KMO=0.895, por lo que la matriz es adecuada. El análisis paralelo recomienda **cuatro factores**, coincidiendo con Blend. La solución rotada permite reconocer las cuatro áreas teóricas, especialmente diseño experimental y estadística/enseñanza.

Sin embargo, hay cargas cruzadas (por ejemplo q1, q9 y q27) e ítems débiles (como q12), y la dimensión interpersonal no queda completamente limpia. **La teoría de Blend recibe apoyo general respecto del número y contenido de factores, pero no apoyo perfecto a nivel de todos los ítems.** Se recomienda revisar o eliminar ítems ambiguos antes de una validación confirmatoria.